# 05 — What the models learned: SHAP values for every test nowcast

**Reads:** `data/processed/features.csv`, `data/processed/nowcasts.csv`, `results/tables/cv_choice.json`.

**Writes:** `results/tables/shap_values.csv`, `shap_importance.csv`, `shap_block_importance.csv`,
`shap_2021.csv`.

A SHAP value splits one nowcast into a starting point (the model's average nowcast over its training months)
plus one contribution per input, in percentage points, and the contributions add up exactly to the
nowcast. Each test nowcast is explained with the model that made it (the one refitted that month), so the
numbers describe what the model actually did in real time.

Random forest: `shap.TreeExplainer`. XGBoost: XGBoost's own TreeSHAP (`pred_contribs=True`), which gives the
same exact values without depending on the `shap` package's XGBoost parser. Linear regression: an input's
contribution is its coefficient times its distance from the training average, which is the exact SHAP value
for a linear model.

SHAP describes how the model used its inputs, not what causes inflation.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
for d in (PROC, TAB, FIG):
    d.mkdir(parents=True, exist_ok=True)

import shap
import xgboost as xgb
from nowcast_setup import TEST_START, FEATURE_SETS, FEATURE_LABELS, MODEL_LABELS, make_model, load_features

feat = load_features(PROC / "features.csv")
choice = json.loads((TAB / "cv_choice.json").read_text())
now = pd.read_csv(PROC / "nowcasts.csv")
now["period"] = pd.PeriodIndex(now["period"], freq="M")
now = now.set_index("period")
test = feat.loc[TEST_START:]

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
records = []
for kind in ("ols", "rf", "xgb"):
    cols = FEATURE_SETS[choice[kind]["feature_set"]]
    for t in test.index:
        train = feat.loc[: t - 1]
        model = make_model(kind, choice[kind]["params"]).fit(train[cols], train["y"])
        x = test.loc[[t], cols]
        pred = float(model.predict(x)[0])
        assert abs(pred - now.at[t, kind]) < 1e-6, (kind, t)       # same model as notebook 04
        if kind == "ols":
            mean_x = train[cols].mean()
            values = model.coef_ * (x.iloc[0] - mean_x).to_numpy()
            base = float(model.intercept_ + model.coef_ @ mean_x.to_numpy())
        elif kind == "rf":
            explainer = shap.TreeExplainer(model)
            values = np.asarray(explainer.shap_values(x))[0]
            base = float(np.ravel(explainer.expected_value)[0])
        else:
            contrib = model.get_booster().predict(xgb.DMatrix(x), pred_contribs=True)[0]
            values, base = contrib[:-1], float(contrib[-1])
        assert abs(base + values.sum() - pred) < 1e-4, (kind, t)    # contributions add up to the nowcast
        records.append({"period": str(t), "model": kind, "input": "starting point (average nowcast)",
                        "value": np.nan, "shap": base})
        for c, v in zip(cols, values):
            records.append({"period": str(t), "model": kind, "input": c, "value": float(x[c].iloc[0]),
                            "shap": float(v)})
sv = pd.DataFrame(records)
sv.round(6).to_csv(TAB / "shap_values.csv", index=False)
sv.head(10)

,period,model,input,value,shap
0,2021-01,ols,starting point (average nowcast),NaN,0.155026
1,2021-01,ols,core_last,0.100000,-0.024224
2,2021-01,ols,core_avg3,0.100000,0.010220
3,2021-01,ols,core_avg12,0.125000,-0.002352
4,2021-01,ols,manheim_0,1.225734,0.012863
5,2021-01,ols,manheim_1,-0.593765,-0.004836
6,2021-01,ols,manheim_2,0.074276,-0.000339
7,2021-02,ols,starting point (average nowcast),NaN,0.154211
8,2021-02,ols,core_last,0.000000,-0.066631
9,2021-02,ols,core_avg3,0.100000,0.009911


## Average size of each input's contribution over the test months

In [3]:
inputs = sv[sv["input"].isin(FEATURE_LABELS)]
importance = (inputs.assign(abs_shap=inputs["shap"].abs())
                    .pivot_table(index="input", columns="model", values="abs_shap", aggfunc="mean")
                    .rename(columns=MODEL_LABELS))
importance.index = [FEATURE_LABELS[i] for i in importance.index]
importance.round(4).to_csv(TAB / "shap_importance.csv")
importance.round(3)

model,Linear regression,Random forest,Gradient boosting (XGBoost)
"Core CPI, average of the last 12 prints",0.086,0.058,0.071
"Core CPI, average of the last 3 prints",0.020,0.020,0.003
"Core CPI, last month's print",0.071,0.031,0.042
"Jet fuel, same month",NaN,0.004,0.006
"Jet fuel, one month earlier",NaN,0.003,0.003
"Used-car prices, same month",0.033,NaN,NaN
"Used-car prices, one month earlier",0.018,NaN,NaN
"Used-car prices, two months earlier",0.014,NaN,NaN


In [4]:
block_of = lambda name: {"core": "Core CPI history", "manheim": "Used-car prices", "jet": "Jet fuel"}[name.split("_")[0]]
blocks = (inputs.assign(block=inputs["input"].map(block_of))
                .groupby(["model", "period", "block"])["shap"].sum().abs()
                .groupby(level=["model", "block"]).mean().unstack("model").rename(columns=MODEL_LABELS))
blocks.round(4).to_csv(TAB / "shap_block_importance.csv")
blocks.round(3)

model,Linear regression,Random forest,Gradient boosting (XGBoost)
block,,,
Core CPI history,0.134,0.097,0.108
Jet fuel,NaN,0.006,0.007
Used-car prices,0.053,NaN,NaN


## The 2021 nowcasts, input by input

In [5]:
y2021 = sv[sv["period"].str.startswith("2021")].copy()
y2021["input"] = y2021["input"].map(lambda i: FEATURE_LABELS.get(i, i))
wide = y2021.pivot_table(index=["period", "input"], columns="model", values="shap").rename(columns=MODEL_LABELS)
wide.round(4).to_csv(TAB / "shap_2021.csv")
wide.loc["2021-04"].round(3)

model,Linear regression,Random forest,Gradient boosting (XGBoost)
input,,,
"Core CPI, average of the last 12 prints",-0.002,0.001,0.001
"Core CPI, average of the last 3 prints",0.004,0.001,0.000
"Core CPI, last month's print",0.062,0.016,0.017
"Jet fuel, one month earlier",NaN,0.002,0.001
"Jet fuel, same month",NaN,0.004,0.005
"Used-car prices, one month earlier",0.035,NaN,NaN
"Used-car prices, same month",0.104,NaN,NaN
"Used-car prices, two months earlier",0.007,NaN,NaN
starting point (average nowcast),0.155,0.154,0.154


In [6]:
now.loc["2021-04", ["y", "poll", "ols", "rf", "xgb", "average"]].astype(float).round(2)

y          0.90
poll       0.30
ols        0.37
rf         0.18
xgb        0.18
average    0.18
Name: 2021-04, dtype: float64

## The linear regression's coefficients, fitted on all months before 2021

The coefficient is the change in the nowcast (percentage points) for a one-unit change in the input (one
percentage point of core CPI, or one percent change in Manheim's index). The second column scales it by one
standard deviation of the input, so the inputs can be compared.

In [7]:
cols = FEATURE_SETS[choice["ols"]["feature_set"]]
train = feat.loc[: TEST_START - 1]
ols = make_model("ols", {}).fit(train[cols], train["y"])
coef = pd.DataFrame({"coefficient": ols.coef_,
                     "effect of a one-standard-deviation move": ols.coef_ * train[cols].std().to_numpy()},
                    index=[FEATURE_LABELS[c] for c in cols])
coef.loc["intercept"] = [ols.intercept_, np.nan]
coef.round(4).to_csv(TAB / "ols_coefficients_before_2021.csv")
coef.round(3)

,coefficient,effect of a one-standard-deviation move
"Core CPI, last month's print",0.428,0.046
"Core CPI, average of the last 3 prints",-0.178,-0.014
"Core CPI, average of the last 12 prints",0.071,0.002
"Used-car prices, same month",0.013,0.023
"Used-car prices, one month earlier",0.006,0.010
"Used-car prices, two months earlier",0.002,0.003
intercept,0.100,NaN
